# 15 — LASSO Feature Selection

Screens the country-year feature panel from Notebook 14 using L1-regularised logistic
regression (LASSO), then rescues non-linearly associated features that LASSO would
silently discard.

## Method

1. **LASSO screening** — `LogisticRegressionCV(penalty='l1', solver='saga')` with 5
   expanding temporal folds inside the training window (2000–2018). λ is chosen by
   the 1-SE rule (most regularised λ within 1 standard error of best AUPRC).
2. **Mutual information rescue** — `mutual_info_classif` detects features with
   U-shaped, threshold, or interaction-only effects that receive a zero LASSO
   coefficient despite having real predictive information. Features in the MI top-50
   that were zeroed by LASSO and show a large MI-vs-LASSO rank gap are rescued.
3. **Audit table** — per feature: LASSO coefficient, MI score, LASSO rank, MI rank,
   rank gap, rescued flag. Written to ADLS for use in Notebook 17.

## Outputs per outcome
- `feature_selection/{RUN_DATE}/selected_{outcome}.json` — feature manifest
- `feature_selection/{RUN_DATE}/audit_{outcome}.parquet` — audit table
- MLflow: regularisation path plot (λ vs. coefficient magnitude)

## Required environment variables
```
ADLS_ACCOUNT_NAME
ADLS_CONTAINER       (default: 'data')
AZUREML_MLFLOW_URI   (optional, for MLflow tracking)
```

In [ ]:
import os
import json
import re
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import rankdata

from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import BaseCrossValidator

from azure.identity import DefaultAzureCredential
import adlfs
import mlflow

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 40)

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

TRAIN_END_YEAR    = 2018   # must match notebook 14

OUTCOMES = [
    "civil_war_onset",
    "coup_attempt",
    "regime_backsliding",
    "mass_unrest_onset",
    "humanitarian_crisis_onset",
]

# Mutual information rescue thresholds
MI_TOP_N       = 50   # feature must be in MI top-N to be rescue-eligible
MI_RANK_GAP    = 30   # MI rank must exceed LASSO rank by at least this much

LASSO_N_CS     = 30   # number of regularisation strengths to try
LASSO_MAX_ITER = 5000
RANDOM_STATE   = 42

print(f"Run date       : {RUN_DATE}")
print(f"Train window   : ≤{TRAIN_END_YEAR}")
print(f"Outcomes       : {OUTCOMES}")

## ADLS helpers

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential":   credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

def read_latest_parquet(prefix: str) -> pd.DataFrame | None:
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    full_prefix = f"{ADLS_CONTAINER}/{prefix}"
    try:
        entries = fs.ls(full_prefix, detail=False)
    except FileNotFoundError:
        print(f"  WARNING: prefix not found: {full_prefix}")
        return None
    date_dirs = sorted(
        [e for e in entries if re.search(r'/\d{8}(/|$)', e)],
        reverse=True,
    )
    if not date_dirs:
        print(f"  WARNING: no date partitions found under {full_prefix}")
        return None
    latest_dir = date_dirs[0]
    parquet_files = [
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
        + f.replace(f"{ADLS_CONTAINER}/", "", 1)
        for f in fs.glob(f"{latest_dir}/*.parquet")
    ]
    if not parquet_files:
        print(f"  WARNING: no .parquet files in {latest_dir}")
        return None
    dfs = [pd.read_parquet(p, storage_options=storage_options) for p in parquet_files]
    df = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
    print(f"  Loaded {len(df):,} rows from {latest_dir}")
    return df

def write_json(obj: dict, subpath: str) -> None:
    import io
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    path = f"{ADLS_CONTAINER}/{subpath}"
    with fs.open(path, "w") as f:
        json.dump(obj, f, indent=2)
    print(f"  Written JSON → {path}")

## Load feature matrix and labels

In [ ]:
df_features = read_latest_parquet("processed/feature_matrix")
df_labels   = read_latest_parquet("processed/feature_matrix")  # same partition

# labels.parquet is a separate file in the same directory — reload it explicitly
if df_features is not None:
    # Re-read labels separately (read_latest_parquet concatenates all parquets
    # in the partition; we need to separate features from labels)
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    prefix = f"{ADLS_CONTAINER}/processed/feature_matrix"
    entries = fs.ls(prefix, detail=False)
    date_dirs = sorted(
        [e for e in entries if re.search(r'/\d{8}(/|$)', e)], reverse=True
    )
    if date_dirs:
        latest = date_dirs[0]
        def _read_one(filename):
            files = fs.glob(f"{latest}/{filename}")
            if not files:
                return None
            path = (
                f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
                + files[0].replace(f"{ADLS_CONTAINER}/", "", 1)
            )
            return pd.read_parquet(path, storage_options=storage_options)

        df_features = _read_one("feature_matrix.parquet")
        df_labels   = _read_one("labels.parquet")

OUTCOME_COLS = [
    "civil_war_onset", "coup_attempt", "regime_backsliding",
    "mass_unrest_onset", "humanitarian_crisis_onset",
]
ID_COLS = ["iso3", "year"]

if df_features is not None and df_labels is not None:
    feat_cols = [c for c in df_features.columns if c not in ID_COLS + OUTCOME_COLS]
    print(f"Feature matrix : {df_features.shape}")
    print(f"Labels         : {df_labels.shape}")
    print(f"Feature columns: {len(feat_cols)}")
else:
    print("ERROR: could not load feature matrix or labels from ADLS")

## Temporal cross-validation splitter

Five expanding-window folds, all within the training period (2000–2018).
No shuffling across time — each fold's validation set is strictly later than its train set.

In [ ]:
class ExpandingYearCV(BaseCrossValidator):
    """
    Expanding-window temporal CV on a country-year panel.

    The training window grows year by year; each fold's validation set is the
    next `val_years` years. Only folds where the validation set has at least
    `min_val_positives` positive examples are kept.
    """

    def __init__(self, years: np.ndarray, n_splits: int = 5, val_years: int = 2,
                 min_val_positives: int = 2):
        self.years           = years
        self.n_splits        = n_splits
        self.val_years       = val_years
        self.min_val_positives = min_val_positives

    def split(self, X, y=None, groups=None):
        unique_years = np.sort(np.unique(self.years))
        # Candidate cutpoints: all but the last val_years years
        candidates = unique_years[: -self.val_years]
        # Evenly space n_splits cutpoints across candidates
        step = max(1, len(candidates) // self.n_splits)
        cutpoints = candidates[step - 1 :: step][: self.n_splits]

        for cutpoint in cutpoints:
            train_idx = np.where(self.years <= cutpoint)[0]
            val_idx   = np.where(
                (self.years > cutpoint) &
                (self.years <= cutpoint + self.val_years)
            )[0]
            if y is not None and len(val_idx) > 0:
                n_pos = np.sum(np.array(y)[val_idx] == 1)
                if n_pos < self.min_val_positives:
                    continue
            if len(train_idx) and len(val_idx):
                yield train_idx, val_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

print("ExpandingYearCV defined")

## Data preparation and LASSO fitting helpers

In [ ]:
def prepare_Xy(outcome: str, fews_countries: set | None = None):
    """
    Build X (features) and y (outcome) for LASSO fitting.

    Follows the same imputation pattern as notebook 16's prepare_outcome_df():
    - Keeps all rows before computing imputation statistics
    - Evaluates column validity (all-NA, zero-variance) on training rows only
    - Computes median imputation values from training rows only
    - Applies those fixed medians to all rows, then returns the training subset

    This prevents val/test data from informing imputation statistics.
    """
    if df_features is None or df_labels is None:
        raise RuntimeError("Feature matrix / labels not loaded")

    # Merge on iso3 + year — keep ALL years at this stage
    merged = df_features.merge(
        df_labels[["iso3", "year", outcome]],
        on=["iso3", "year"], how="inner",
    )

    # For humanitarian outcome, restrict to FEWS countries
    if outcome == "humanitarian_crisis_onset" and fews_countries:
        merged = merged[merged["iso3"].isin(fews_countries)]

    # Drop rows with NA label
    merged = merged[merged[outcome].notna()].copy()

    feat_cols = [c for c in df_features.columns if c not in ["iso3", "year"] + OUTCOME_COLS]

    # Training mask — all statistics derived exclusively from this subset
    train_mask = merged["year"] <= TRAIN_END_YEAR

    # Drop columns that are all-NA or zero-variance *in training rows* (matches nb16)
    X_tr_check = merged.loc[train_mask, feat_cols]
    feat_cols = [
        c for c in feat_cols
        if X_tr_check[c].notna().any() and X_tr_check[c].nunique() > 1
    ]

    # Median impute using training rows only, applied to all rows (matches nb16)
    medians = merged.loc[train_mask, feat_cols].median()
    merged[feat_cols] = merged[feat_cols].fillna(medians)

    # Now restrict to training period for LASSO fitting
    merged_train = merged[train_mask].copy()

    X         = merged_train[feat_cols].values
    y         = merged_train[outcome].astype(int).values
    years_arr = merged_train["year"].values

    n_pos = int(y.sum())
    print(f"{outcome}: {len(y):,} training rows | {n_pos} positives "
          f"({y.mean():.3f}) | {len(feat_cols)} features")
    return X, y, feat_cols, years_arr


def fit_lasso(X_train, y_train, years_arr, feature_names):
    """
    Fit L1 logistic regression with temporal CV and the 1-SE rule.

    Returns:
        coef       — array of shape (n_features,), best-λ coefficients
        lambda_1se — the chosen λ value
        Cs_grid    — the C grid used (C = 1/λ)
        cv_scores  — mean AUPRC per C across folds (for path plot)
        scaler     — fitted StandardScaler (for external use)
    """
    cv = ExpandingYearCV(years=years_arr, n_splits=5, val_years=2, min_val_positives=2)

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    lasso_cv = LogisticRegressionCV(
        penalty="l1",
        solver="saga",
        Cs=LASSO_N_CS,
        cv=cv,
        scoring="average_precision",
        max_iter=LASSO_MAX_ITER,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    lasso_cv.fit(X_scaled, y_train)

    # lasso_cv.scores_[1]: shape (n_folds, n_Cs) — AUPRC per fold per C
    fold_scores = lasso_cv.scores_[1]
    mean_scores = fold_scores.mean(axis=0)
    std_scores  = fold_scores.std(axis=0)

    best_idx   = np.argmax(mean_scores)
    best_score = mean_scores[best_idx]
    best_std   = std_scores[best_idx]

    # 1-SE rule: most regularised C (smallest C = largest λ) within 1 SE of best
    threshold  = best_score - best_std
    eligible   = np.where(mean_scores >= threshold)[0]
    chosen_idx = eligible[0]   # smallest C still within 1 SE

    chosen_C   = lasso_cv.Cs_[chosen_idx]
    lambda_1se = 1.0 / chosen_C

    # lasso_cv.coefs_paths_[1]: shape (n_folds, n_Cs, n_features)
    coef = lasso_cv.coefs_paths_[1].mean(axis=0)[chosen_idx]

    print(f"  Best C={lasso_cv.Cs_[best_idx]:.5f}  1SE C={chosen_C:.5f}  "
          f"λ_1se={lambda_1se:.5f}  non-zero={int((coef != 0).sum())}")

    return coef, lambda_1se, lasso_cv.Cs_, mean_scores, scaler

print("prepare_Xy and fit_lasso defined")

## Mutual information rescue and audit table

In [ ]:
def build_audit_table(feature_names, lasso_coef, X_train, y_train):
    """
    Build the LASSO vs. MI audit table and identify rescued features.

    A feature is rescued if:
      - LASSO zeroed it (coef == 0)
      - MI rank is within top MI_TOP_N
      - MI rank exceeds LASSO rank by at least MI_RANK_GAP
        (large gap = MI says important but LASSO says not)
    """
    mi_scores = mutual_info_classif(
        X_train, y_train, random_state=RANDOM_STATE
    )

    lasso_rank = rankdata(-np.abs(lasso_coef), method="min")
    mi_rank    = rankdata(-mi_scores,          method="min")

    audit = pd.DataFrame({
        "feature":       feature_names,
        "lasso_coef":    lasso_coef,
        "lasso_nonzero": lasso_coef != 0,
        "mi_score":      mi_scores,
        "lasso_rank":    lasso_rank.astype(int),
        "mi_rank":       mi_rank.astype(int),
    })
    audit["rank_gap"] = audit["mi_rank"] - audit["lasso_rank"]

    # Rescue criterion
    audit["rescued"] = (
        (~audit["lasso_nonzero"])
        & (audit["mi_rank"] <= MI_TOP_N)
        & (audit["rank_gap"] > MI_RANK_GAP)
    )

    return audit.sort_values("lasso_rank")


def plot_regularisation_path(Cs, cv_scores, outcome, lambda_1se):
    """Return a matplotlib Figure of AUPRC vs. log(λ)."""
    lambdas = 1.0 / Cs
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.semilogx(lambdas, cv_scores, "b-o", markersize=3)
    ax.axvline(lambda_1se, color="red", linestyle="--", label=f"λ_1SE={lambda_1se:.4f}")
    ax.set_xlabel("λ (1/C)")
    ax.set_ylabel("Mean AUPRC (temporal CV)")
    ax.set_title(f"LASSO regularisation path — {outcome}")
    ax.legend()
    fig.tight_layout()
    return fig

print("build_audit_table and plot_regularisation_path defined")

## Run LASSO selection for all outcomes

Iterates over the 5 outcomes, fits LASSO, computes MI rescue, and writes:
- `feature_selection/{RUN_DATE}/selected_{outcome}.json`
- `feature_selection/{RUN_DATE}/audit_{outcome}.parquet`

In [ ]:
# Load FEWS country set from crosswalk (needed for humanitarian outcome)
from pathlib import Path
_cw_path = Path("../data/country_crosswalk.csv")
if not _cw_path.exists():
    _cw_path = Path("data/country_crosswalk.csv")
_cw = pd.read_csv(_cw_path, dtype=str)
FEWS_COUNTRIES = set(_cw.loc[_cw["fews_monitored"] == "1", "iso3"])

# Optional MLflow tracking
mlflow_uri = os.getenv("AZUREML_MLFLOW_URI", "")
if mlflow_uri:
    mlflow.set_tracking_uri(mlflow_uri)
    mlflow.set_experiment("feature_selection_lasso")

results = {}

for outcome in OUTCOMES:
    print(f"\n{'='*60}")
    print(f"Outcome: {outcome}")
    print('='*60)

    fews = FEWS_COUNTRIES if outcome == "humanitarian_crisis_onset" else None

    try:
        X_train, y_train, feat_names, years_arr = prepare_Xy(outcome, fews)
    except Exception as e:
        print(f"  SKIPPED — {e}")
        continue

    if y_train.sum() < 5:
        print(f"  SKIPPED — too few positives ({int(y_train.sum())})")
        continue

    # Fit LASSO with temporal CV
    coef, lambda_1se, Cs, cv_scores, scaler = fit_lasso(
        X_train, y_train, years_arr, feat_names
    )

    # Build audit table
    audit = build_audit_table(feat_names, coef, X_train, y_train)
    audit["outcome"] = outcome

    lasso_features   = audit.loc[audit["lasso_nonzero"],  "feature"].tolist()
    rescued_features = audit.loc[audit["rescued"],         "feature"].tolist()
    all_features     = sorted(set(lasso_features + rescued_features))

    print(f"  LASSO selected : {len(lasso_features)}")
    print(f"  MI rescued     : {len(rescued_features)}")
    print(f"  Total          : {len(all_features)}")

    # Feature manifest JSON
    manifest = {
        "outcome":            outcome,
        "run_date":           RUN_DATE,
        "lasso_lambda_1se":   float(lambda_1se),
        "n_features_lasso":   len(lasso_features),
        "n_features_rescued": len(rescued_features),
        "n_features_total":   len(all_features),
        "feature_names":      all_features,
        "lasso_features":     lasso_features,
        "rescued_features":   rescued_features,
    }
    write_json(manifest, f"feature_selection/{RUN_DATE}/selected_{outcome}.json")

    # Audit parquet
    write_parquet(audit, f"feature_selection/{RUN_DATE}/audit_{outcome}.parquet")

    # Regularisation path plot
    fig = plot_regularisation_path(Cs, cv_scores, outcome, lambda_1se)

    if mlflow_uri:
        with mlflow.start_run(run_name=outcome, tags={"outcome": outcome}):
            mlflow.log_params({
                "lasso_lambda_1se":   lambda_1se,
                "n_features_lasso":   len(lasso_features),
                "n_features_rescued": len(rescued_features),
            })
            mlflow.log_figure(fig, f"lasso_path_{outcome}.png")

    plt.show()
    plt.close(fig)

    results[outcome] = manifest

print(f"\n{'='*60}")
print("Feature selection complete")
for outcome, m in results.items():
    print(f"  {outcome:<35} {m['n_features_total']:3d} features")